In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# ---------------------------------------------------------
# 1. Datos de entrada
# ---------------------------------------------------------
asset_names = ['Stocks', 'Bonds', 'Real E.']

E = np.array([0.065, 0.040, 0.050])
sigma = np.array([0.16, 0.07, 0.12])

Omega = np.array([
    [0.02560, 0.00392, 0.00384],
    [0.00392, 0.00490, 0.00168],
    [0.00384, 0.00168, 0.01440]
])

ones = np.ones(len(E))
Omega_inv = np.linalg.inv(Omega)

# Tasa libre de riesgo (Rf)
rf = 0.030  # 3.0%

In [8]:
# ---------------------------------------------------------
# 2. Cálculo del Portafolio de Tangencia (Máximo Sharpe Ratio)
# ---------------------------------------------------------
excess_returns = E - rf * ones
w_tangency_unscaled = Omega_inv @ excess_returns
w_tangency = w_tangency_unscaled / np.sum(w_tangency_unscaled)

# Retorno y Volatilidad del Portafolio de Tangencia
E_tangency = float(w_tangency.T @ E)
sigma_tangency = float(np.sqrt(w_tangency.T @ Omega @ w_tangency))
sharpe_ratio = (E_tangency - rf) / sigma_tangency

In [9]:
# ---------------------------------------------------------
# 3. Línea del Mercado de Capitales (CML)
# ---------------------------------------------------------
# CML: E(Rp) = Rf + Sharpe * sigma_p
sigma_cml = np.linspace(0, 0.25, 200)
E_cml = rf + sharpe_ratio * sigma_cml

In [10]:
# ---------------------------------------------------------
# 4. Frontera Eficiente de Markowitz
# ---------------------------------------------------------
a11 = float(E.T @ Omega_inv @ E)
a12 = float(ones.T @ Omega_inv @ E)
a22 = float(ones.T @ Omega_inv @ ones)

A = np.array([[a11, a12], [a12, a22]])
delta = np.linalg.det(A)

mu_vals = np.linspace(0.01, 0.12, 500)
sigma2_p = (a22 / delta) * (mu_vals - (a12 / a22))**2 + (1 / a22)
sigma_p = np.sqrt(sigma2_p)

mu_min_var = a12 / a22
sigma_min_var = np.sqrt(1 / a22)

# Imprimir resultados del Portafolio de Tangencia
print("=" * 55)
print("             PORTAFOLIO DE TANGENCIA Y CAPM              ")
print("=" * 55)
print(f"Tasa Libre de Riesgo (Rf)  : {rf:.2%}")
print(f"Sharpe Ratio Máximo        : {sharpe_ratio:.4f}")
print(f"Retorno Esp. Tangencia (E) : {E_tangency:.2%}")
print(f"Volatilidad Tangencia (σ)  : {sigma_tangency:.2%}\n")
print("Pesos del Portafolio de Tangencia:")
for name, w in zip(asset_names, w_tangency):
    print(f"  - {name:10s}: {w:7.2%}")
print("=" * 55)

             PORTAFOLIO DE TANGENCIA Y CAPM              
Tasa Libre de Riesgo (Rf)  : 3.00%
Sharpe Ratio Máximo        : 0.2578
Retorno Esp. Tangencia (E) : 5.28%
Volatilidad Tangencia (σ)  : 8.84%

Pesos del Portafolio de Tangencia:
  - Stocks    :  37.38%
  - Bonds     :  28.27%
  - Real E.   :  34.34%


In [21]:
# ---------------------------------------------------------
# 5. Visualización con Plotly
# ---------------------------------------------------------
fig = go.Figure()

# Frontera de Mínima Varianza (completa)
fig.add_trace(go.Scatter(
    x=sigma_p, y=mu_vals,
    mode='lines',
    name='Frontera de Mínima Varianza',
    line=dict(color='gray', dash='dash', width=1.5),
    hovertemplate='σ: %{x:.2%}<br>E: %{y:.2%}'
))

# Frontera Eficiente
mask_efficient = mu_vals >= mu_min_var
fig.add_trace(go.Scatter(
    x=sigma_p[mask_efficient], y=mu_vals[mask_efficient],
    mode='lines',
    name='Frontera Eficiente (Sin Rf)',
    line=dict(color='#00CC96', width=3),
    hovertemplate='σ: %{x:.2%}<br>E: %{y:.2%}'
))

# Línea del Mercado de Capitales (CML)
fig.add_trace(go.Scatter(
    x=sigma_cml, y=E_cml,
    mode='lines',
    name='Línea Mercado Capitales (CML)',
    line=dict(color='#FFD700', width=2, dash='dot'),
    hovertemplate='σ: %{x:.2%}<br>E: %{y:.2%}'
))

# Activo Libre de Riesgo (Rf)
fig.add_trace(go.Scatter(
    x=[0], y=[rf],
    mode='markers+text',
    name='Activo Libre de Riesgo (Rf)',
    text=['Rf (3%)'],
    textposition='top right',
    marker=dict(size=10, color='white', symbol='circle', line=dict(width=0, color='orange')),
    hovertemplate='<b>Rf</b><br>Retorno: %{y:.2%}'
))

# Portafolio de Tangencia
fig.add_trace(go.Scatter(
    x=[sigma_tangency], y=[E_tangency],
    mode='markers+text',
    name='Portafolio de Tangencia',
    text=['MVE Portafolio'],
    textposition='top left',
    marker=dict(size=12, color='white', symbol='star', line=dict(width=0, color='white')),
    hovertemplate=f'<b>Portafolio de Tangencia</b><br>σ: %{{x:.2%}}<br>E: %{{y:.2%}}<br>Sharpe: {sharpe_ratio:.2f}'
))

# GMVP
fig.add_trace(go.Scatter(
    x=[sigma_min_var], y=[mu_min_var],
    mode='markers',
    name='GMVP',
    marker=dict(size=10, color='mediumpurple', symbol='star'),
    hovertemplate='<b>GMVP</b><br>σ: %{x:.2%}<br>E: %{y:.2%}'
))

# Activos individuales
fig.add_trace(go.Scatter(
    x=sigma, y=E,
    mode='markers+text',
    name='Activos Individuales',
    text=asset_names,
    textposition='top center',
    marker=dict(size=9, color='crimson'),
    hovertemplate='<b>%{text}</b><br>σ: %{x:.2%}<br>E: %{y:.2%}'
))

# Layout
fig.update_layout(
    title='<b>Frontera Eficiente de Markowitz y Capital Market Line (CML)</b>',
    xaxis_title='Riesgo / Volatilidad (σ)',
    yaxis_title='Retorno Esperado (E)',
    xaxis=dict(tickformat='.1%', range=[-0.01, 0.22]),
    yaxis=dict(tickformat='.1%', range=[0.01, 0.11]),
    template='plotly_dark',
    hovermode='closest',
    width=900,
    height=600
)

fig.show()